In [ ]:
# ===== CELL 1: imports, configuration, device + GPU memory cap =====
import os, re, io, sys, json, time, math, gc, zipfile, random, heapq, subprocess, warnings
from pathlib import Path
from collections import Counter, defaultdict
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
warnings.filterwarnings("ignore")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

PROJECT_DIR = Path.cwd()          # the folder you opened in VS Code
ZIP_PATH = None                   # None = auto-detect first *.zip in project folder (or ./data); else e.g. "dataset.zip"
DATA_DIR, OUT_DIR, CACHE_DIR = PROJECT_DIR / "data", PROJECT_DIR / "outputs", PROJECT_DIR / "cache"
ORDERED_LOCAL_DIR = PROJECT_DIR / "ordered_hindi"     # optional fallback corpus: *.txt, one paragraph per line
for d in (DATA_DIR, OUT_DIR, CACHE_DIR): d.mkdir(exist_ok=True)

CFG = dict(
    seed=42,
    use_gpu=True, gpu_budget_gb=0.8,     # PyTorch allocator cap. nvidia-smi also counts the CUDA context (~0.3-0.5 GB) -> keep this <= 1.0
    punct_model="Horizon-Labs/punctuation-restoration-small",
    label_map_override=None,             # only if id2label is generic, e.g. {0:"",1:".",2:",",3:"?",4:"-",5:":"}
    punct_win_words=100, punct_stride_words=70,
    use_danda=True,                      # write '.' as Hindi danda '।' in outputs (internal logic unaffected)
    soft_max_words=30, hard_max_words=45, min_side_words=5, split_score_thresh=0.6, null_bonus=0.3,
    force_final_period=True,
    enc_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    n_train_passages=6000, n_heldout_passages=300,   # lower n_train_passages (e.g. 3000) for a faster CPU-only run
    train_steps=4000, batch=256, lr=1e-3,
    restarts=6,
    run_weight_search=True,
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
torch.set_num_threads(max(1, min(8, os.cpu_count() or 4)))

DEVICE = "cpu"
if CFG["use_gpu"] and torch.cuda.is_available():
    try:
        _x = torch.ones(64, 64, device="cuda"); float((_x @ _x).sum())      # proves kernels exist for this GPU
        total = torch.cuda.get_device_properties(0).total_memory
        torch.cuda.set_per_process_memory_fraction(min(0.95, CFG["gpu_budget_gb"] * 1024**3 / total))
        DEVICE = "cuda"; del _x
    except Exception as ex:
        print("CUDA present but unusable (", str(ex)[:120], ") -> falling back to CPU. See the setup notes (cu128/cu130 wheel).")
TRAIN_DEVICE = "cpu"              # the ordering head is tiny; training on CPU keeps the GPU for the two encoders only

def free_gpu():
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
def gpu_report(tag=""):
    if DEVICE != "cuda": print(f"[{tag}] CPU mode"); return
    print(f"[{tag}] torch allocated {torch.cuda.memory_allocated()/2**30:.2f} GiB | peak allocated {torch.cuda.max_memory_allocated()/2**30:.2f} GiB | "
          f"reserved {torch.cuda.memory_reserved()/2**30:.2f} GiB")
    try:
        print("   nvidia-smi:", subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"], text=True).strip())
    except Exception: pass
print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no usable GPU")
if DEVICE == "cuda": print("compute capability:", torch.cuda.get_device_capability(0), "| allocator cap (GiB):", CFG["gpu_budget_gb"])

In [ ]:
# ===== CELL 2: ZIP extraction, loading, normalisation (KAGGLE) =====

from pathlib import Path
import zipfile
import re

# Kaggle directories
KAGGLE_INPUT = Path("/kaggle/input")
DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Find uploaded ZIP automatically
def find_zip():
    if ZIP_PATH is not None:
        return Path(ZIP_PATH)

    # Search all Kaggle input datasets
    zips = sorted(KAGGLE_INPUT.rglob("*.zip"))
    return zips[0] if zips else None


zp = find_zip()
EXTRACT_DIR = DATA_DIR / "extracted"

if zp is not None:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zp) as z:
        z.extractall(EXTRACT_DIR)

    print("Extracted:", zp)

else:
    print("No ZIP found. Searching Kaggle input for sample-*.txt")
    EXTRACT_DIR = KAGGLE_INPUT


# Find all sample files
found = {}

for p in EXTRACT_DIR.rglob("sample-*.txt"):
    if "__MACOSX" in p.parts:
        continue

    m = re.search(r"sample-(\d+)\.txt$", p.name)

    if m:
        found[int(m.group(1))] = p


if not found:
    raise FileNotFoundError(
        "No sample-*.txt found. Check your Kaggle dataset upload."
    )


FILENAMES = [f"sample-{k}.txt" for k in sorted(found)]

if len(FILENAMES) != 7:
    print("WARNING: expected 7 files, found", len(FILENAMES), FILENAMES)


# Read text files
def read_text(p):
    for enc in ("utf-8-sig", "utf-16"):
        try:
            return p.read_text(encoding=enc)
        except UnicodeError:
            continue

    raise UnicodeError(f"Cannot decode {p}")


RAW = {
    f"sample-{k}.txt": read_text(found[k])
    for k in sorted(found)
}


# ONLY normalisation: collapse whitespace
# NULL and every other character remain untouched
DOCS = {
    fn: " ".join(t.split())
    for fn, t in RAW.items()
}

TOKENS = {
    fn: t.split()
    for fn, t in DOCS.items()
}


# Dataset statistics
print(f"{'file':<14}{'words':>7}{'chars':>8}{'NULL':>6}")

for fn in FILENAMES:
    print(
        f"{fn:<14}"
        f"{len(TOKENS[fn]):>7}"
        f"{len(DOCS[fn]):>8}"
        f"{sum(t == 'NULL' for t in TOKENS[fn]):>6}"
    )

print(
    "TOTAL words",
    sum(len(v) for v in TOKENS.values()),
    "| chars",
    sum(len(v) for v in DOCS.values()),
    "| NULL",
    sum(t == "NULL" for v in TOKENS.values() for t in v),
    "(handoff stated ~5,821 / 28,216 / 67)"
)

odd = [
    (fn, t)
    for fn in FILENAMES
    for t in TOKENS[fn]
    if "NULL" in t and t != "NULL"
]

if odd:
    print("NOTE: tokens that merely contain 'NULL':", odd[:10])

In [ ]:
%pip install -q transformers sentence-transformers datasets tqdm

In [ ]:
# ===== CELL 3: punctuation restoration (Horizon-Labs/punctuation-restoration-small) =====
# Model-card convention: plain token classification; a word's label is the label of its LAST sub-token ("0" = no mark).
# NULL tokens are removed from the model input and re-inserted afterwards (never given punctuation).
from transformers import AutoTokenizer, AutoModelForTokenClassification
_LABEL_TO_CHAR = {"0": "", "O": "", "NONE": "", "NO_PUNCT": "", ".": ".", "PERIOD": ".", ",": ",", "COMMA": ",",
                  "?": "?", "QUESTION": "?", "QUESTION_MARK": "?", "-": "-", "DASH": "-", ":": ":", "COLON": ":", "!": "!", ";": ";"}

tok = AutoTokenizer.from_pretrained(CFG["punct_model"])
mdl = AutoModelForTokenClassification.from_pretrained(CFG["punct_model"]).eval()
id2label = {int(k): str(v) for k, v in mdl.config.id2label.items()}
print("id2label:", id2label)
if CFG["label_map_override"]: label_char = {int(k): v for k, v in CFG["label_map_override"].items()}
else:
    unknown = [v for v in id2label.values() if v.upper() not in _LABEL_TO_CHAR and v not in _LABEL_TO_CHAR]
    if unknown: raise RuntimeError(f"Unrecognised labels {unknown}. Set CFG['label_map_override'] from the model card.")
    label_char = {i: _LABEL_TO_CHAR.get(v, _LABEL_TO_CHAR.get(v.upper())) for i, v in id2label.items()}
n_labels = len(label_char)
none_idx = [i for i, c in label_char.items() if c == ""][0]
term_idx = [i for i, c in label_char.items() if c in {".", "?", "!"}]
if not term_idx: raise RuntimeError("No sentence-ending label recognised.")
mdl = mdl.to(DEVICE, dtype=torch.bfloat16) if DEVICE == "cuda" else mdl
maxlen = int(min(getattr(tok, "model_max_length", 1024) or 1024, 1024))

@torch.no_grad()
def word_probs(words):
    n = len(words); win, stride = CFG["punct_win_words"], CFG["punct_stride_words"]
    probs = np.zeros((n, n_labels), np.float32); probs[:, none_idx] = 1.0; best = np.full(n, -1)
    s, starts = 0, []
    while True:
        starts.append(s)
        if s + win >= n: break
        s += stride
    for s in starts:
        ws = words[s:s + win]; text = " ".join(ws)
        c2w = np.full(len(text), -1, np.int64); pos = 0
        for k, w in enumerate(ws): c2w[pos:pos + len(w)] = k; pos += len(w) + 1
        enc = tok(text, return_offsets_mapping=True, return_tensors="pt", truncation=True, max_length=maxlen)
        offs = enc["offset_mapping"][0].numpy()
        inp = {k: enc[k].to(DEVICE) for k in ("input_ids", "attention_mask") if k in enc}
        pr = torch.softmax(mdl(**inp).logits[0].float(), -1).cpu().numpy()
        last = {}
        for ti, (a, b) in enumerate(offs):
            if b > a and c2w[b - 1] >= 0: last[int(c2w[b - 1])] = ti
        for k, ti in last.items():
            g = s + k
            cent = min(k if s > 0 else 10**6, (len(ws) - 1 - k) if s + len(ws) < n else 10**6)
            if cent > best[g]: best[g] = cent; probs[g] = pr[ti]
    return probs

PUNCT = {}
t0 = time.time()
for fn in FILENAMES:
    toks = TOKENS[fn]; nn_idx = [i for i, t in enumerate(toks) if t != "NULL"]
    P = word_probs([toks[i] for i in nn_idx])
    pred_char = [""] * len(toks); p_term = np.zeros(len(toks), np.float32)
    for r, i in enumerate(nn_idx):
        pred_char[i] = label_char[int(P[r].argmax())]; p_term[i] = P[r, term_idx].sum()
    PUNCT[fn] = dict(tokens=toks, pred_char=pred_char, p_term=p_term)
    print(fn, "words:", len(nn_idx), "| predicted marks:", dict(Counter(c for c in pred_char if c)))
gpu_report("after punctuation inference"); print(f"punctuation inference: {time.time()-t0:.1f}s")
del mdl, tok; free_gpu()

In [ ]:
# ===== CELL 4: text utilities + segmentation repair (pure Python / numpy) =====
PUNCT_CHARS_RE = re.compile(r"[।\.,\?!:;\-–—\"'“”‘’\(\)\[\]\{\}…|]")
INSERTED_RE = re.compile(r"[।\.,\?!:;\-]")

def strip_for_model(s):
    return " ".join(PUNCT_CHARS_RE.sub(" ", s).split())

TERMINALS = {".", "?", "!"}
FINITE_END = set("""है हैं था थे थी थीं हूँ हूं हो होगा होगी होंगे होता होती होते हुआ हुई हुए हुईं
गया गयी गई गए गये गयीं गईं किया किये किए दिया दिये दिए लिया लिये सकता सकती सकते चाहिए चाहिये
रहा रही रहे रहती रहता रहते पड़ा पड़ी पड़े करता करती करते करें जाता जाती जाते जाएगा जाएगी जाएंगे
जायेगा देता देती देते आया आई आए आये मिला मिली मिले बना बनी बने""".split())
ANAPHORIC = set("""यह ये इस इन इसे इसके इसकी इसका इसमें इससे इसी इन्हें इनके इनकी इनका उस उन उसे उसके उसकी
उसका उसमें उससे उसी उन्हें उनके उनकी उनका उनमें उन्होंने उसने वह वे उपरोक्त उपर्युक्त ऐसे ऐसा ऐसी वहाँ
वहां यहाँ यहां इसलिए इसलिये अतः""".split())
CONNECTIVE = set("""लेकिन परंतु परन्तु किंतु किन्तु मगर और तथा एवं फिर तब तभी क्योंकि जबकि हालांकि हालाँकि
वैसे बल्कि यानी अर्थात जैसे""".split())
SENT_STARTERS = ANAPHORIC | CONNECTIVE | set("जब जहाँ जहां अगर यदि अब आज कभी कोई सभी".split())
NONFINAL = set("""के की का को में से पर ने कि जो और तथा एवं या यानी जैसे लिए साथ द्वारा बाद पहले बीच
लेकिन परंतु मगर किंतु जब तब जहाँ जहां जिसके जिनके इस उस इन उन एक यह वह ये वे""".split())

def boundary_score(i, tokens, p_term, e, cfg):
    sc = float(p_term[i])
    if tokens[i] in FINITE_END: sc += 0.25
    if tokens[i] in NONFINAL: sc -= 0.5
    if i + 1 <= e:
        nxt = tokens[i + 1]
        if nxt == "NULL": sc += cfg["null_bonus"]
        elif nxt in SENT_STARTERS: sc += 0.1
    return sc

def render_tokens(tokens, final_punct, s, e, danda):
    out = []
    for i in range(s, e + 1):
        t, c = tokens[i], (final_punct[i] if tokens[i] != "NULL" else "")
        if c == ".": c = "।" if danda else "."
        elif c == "-": c = " -"
        out.append(t + c)
    return " ".join(out)

def segment_file(tokens, pred_char, p_term, cfg):
    n = len(tokens)
    is_null = [t == "NULL" for t in tokens]
    cum = np.cumsum([0 if x else 1 for x in is_null])
    def wc(s, e): return int(cum[e] - (cum[s - 1] if s > 0 else 0))
    final_punct = [("" if is_null[i] else pred_char[i]) for i in range(n)]
    ranges, s = [], 0
    for i in range(n):
        if final_punct[i] in TERMINALS:
            ranges.append((s, i, "model")); s = i + 1
    if s < n: ranges.append((s, n - 1, "eof"))
    repairs = []
    def split_rec(s, e):
        total = wc(s, e)
        if total <= cfg["soft_max_words"]: return [(s, e)]
        best = None
        for i in range(s, e):
            if is_null[i]: continue
            l, r = wc(s, i), wc(i + 1, e)
            if l < cfg["min_side_words"] or r < cfg["min_side_words"]: continue
            sc = boundary_score(i, tokens, p_term, e, cfg) + 0.15 * (2 * min(l, r) / (l + r))
            if best is None or sc > best[0]: best = (sc, i)
        if best is None: return [(s, e)]
        if total > cfg["hard_max_words"] or best[0] >= cfg["split_score_thresh"]:
            sc, i = best
            repairs.append(dict(token_index=i, word=tokens[i], score=round(float(sc), 3),
                                replaced_label=final_punct[i], p_term=round(float(p_term[i]), 3),
                                segment_words_before_split=total))
            final_punct[i] = "."
            return split_rec(s, i) + split_rec(i + 1, e)
        return [(s, e)]
    segs = []
    for (s, e, et) in ranges:
        for (a, b) in split_rec(s, e):
            segs.append([a, b, et if b == e else "repair"])
    # fragments consisting only of NULL tokens are attached to a neighbour (never dropped)
    merged, pending = [], None
    for a, b, et in segs:
        if wc(a, b) == 0:
            if merged: merged[-1][1] = b
            else: pending = a
            continue
        if pending is not None: a, pending = pending, None
        merged.append([a, b, et])
    segs = merged
    if cfg["force_final_period"] and segs:
        a, b, et = segs[-1]
        last = max(i for i in range(a, b + 1) if not is_null[i])
        if final_punct[last] not in TERMINALS:
            final_punct[last] = "."; segs[-1][2] = "eof_period"
    out = []
    for k, (a, b, et) in enumerate(segs):
        words = [t for t in tokens[a:b + 1] if t != "NULL"]
        mtext = strip_for_model(" ".join(words))
        flags = []
        nw = len(words)
        if nw > cfg["hard_max_words"]: flags.append("long")
        if nw < 3: flags.append("short")
        if any(t == "NULL" for t in tokens[a:b + 1]): flags.append("contains_NULL")
        if tokens[a] == "NULL": flags.append("starts_with_NULL")
        if words and words[0] in ANAPHORIC: flags.append("dependent_start")
        elif words and words[0] in CONNECTIVE: flags.append("connective_start")
        if words and words[-1] not in FINITE_END and final_punct[max(i for i in range(a, b + 1) if not is_null[i])] != "?":
            flags.append("no_final_verb")
        out.append(dict(id=k, start=a, end=b, end_type=et, n_words=nw,
                        n_null=sum(is_null[a:b + 1]), flags=flags, model_text=mtext,
                        text=render_tokens(tokens, final_punct, a, b, cfg["use_danda"])))
    return out, final_punct, repairs

def null_contexts(tokens, segs, repairs):
    rep_idx = {r["token_index"] for r in repairs}
    owner = {}
    for sg in segs:
        for i in range(sg["start"], sg["end"] + 1): owner[i] = sg
    res = []
    for j, t in enumerate(tokens):
        if t != "NULL": continue
        sg = owner[j]
        where = "segment_start" if j == sg["start"] else ("segment_end" if j == sg["end"] else "inside_segment")
        res.append(dict(token_index=j, segment_id=sg["id"], where=where,
                        left=" ".join(tokens[max(0, j - 4):j]), right=" ".join(tokens[j + 1:j + 5]),
                        repair_split_just_before=(j - 1) in rep_idx))
    return res

SEGS, FINAL_PUNCT, REPAIRS, AUDIT = {}, {}, {}, {}
for fn in FILENAMES:
    pd_ = PUNCT[fn]
    segs, fp, reps = segment_file(pd_["tokens"], pd_["pred_char"], pd_["p_term"], CFG)
    SEGS[fn], FINAL_PUNCT[fn], REPAIRS[fn] = segs, fp, reps
    nw = np.array([s["n_words"] for s in segs])
    AUDIT[fn] = dict(
        n_tokens=len(pd_["tokens"]), n_words=int(sum(nw)), n_null=int(sum(s["n_null"] for s in segs)),
        n_segments=len(segs),
        n_model_boundaries=sum(1 for s in segs if s["end_type"] == "model"),
        n_repair_splits=len(reps), n_eof_period=sum(1 for s in segs if s["end_type"] == "eof_period"),
        words_per_segment=dict(mean=round(float(nw.mean()), 1), median=float(np.median(nw)), min=int(nw.min()), max=int(nw.max())),
        flag_counts=dict(Counter(f for s in segs for f in s["flags"])),
        repairs=reps, null_contexts=null_contexts(pd_["tokens"], segs, reps))
print(f"{'file':<14}{'segs':>5}{'model':>7}{'repair':>8}{'meanW':>7}{'maxW':>6}{'NULL':>6}  flags")
for fn in FILENAMES:
    a = AUDIT[fn]
    print(f"{fn:<14}{a['n_segments']:>5}{a['n_model_boundaries']:>7}{a['n_repair_splits']:>8}"
          f"{a['words_per_segment']['mean']:>7}{a['words_per_segment']['max']:>6}{a['n_null']:>6}  {a['flag_counts']}")
print("Total segments:", sum(len(v) for v in SEGS.values()))

In [ ]:
# ===== CELL 5: global ordering decoder (pure numpy; objective documented in the report) =====
# Objective for a permutation pi of n segments:
#   J(pi) = sum_{a<b in pi} S[pi_a, pi_b]          pairwise "i before j" evidence over ALL pairs (Kemeny-style)
#         + lam * sum_t A[pi_t, pi_{t+1}]          log P(adjacent) for consecutive items
#         - mu  * sum_t (t/(n-1) - pp[pi_t])^2     soft prior from predicted relative position of each sentence
#         - nu  * dep[pi_0]                        soft penalty if the file opens with a dependent-looking sentence
# subject to hard precedence constraints (optional). Search = insertion + block-move local search with restarts.
# It finds a local optimum of J; J is a learned proxy, NOT a measure of true discourse correctness.

def objective(perm, S, A, pp, dep, lam, mu, nu):
    P = np.asarray(perm); n = len(P)
    pair = np.triu(S[np.ix_(P, P)], 1).sum()
    adj = A[P[:-1], P[1:]].sum() if n > 1 else 0.0
    ranks = np.arange(n) / max(n - 1, 1)
    return pair + lam * adj - mu * ((ranks - pp[P]) ** 2).sum() - nu * dep[P[0]]

def violates(perm, cons):
    if not cons: return False
    pos = {x: i for i, x in enumerate(perm)}
    return any(pos[a] > pos[b] for a, b in cons)

def topo_repair(perm, cons, n):
    if not cons: return list(perm)
    rank = {x: i for i, x in enumerate(perm)}
    succ, indeg = defaultdict(list), [0] * n
    for a, b in cons: succ[a].append(b); indeg[b] += 1
    h = [(rank[x], x) for x in range(n) if indeg[x] == 0]; heapq.heapify(h); out = []
    while h:
        _, x = heapq.heappop(h); out.append(x)
        for y in succ[x]:
            indeg[y] -= 1
            if indeg[y] == 0: heapq.heappush(h, (rank[y], y))
    if len(out) != n: raise ValueError("Constraints contain a cycle")
    return out

def insertion_pass(perm, S, A, pp, dep, lam, mu, nu, pred, succ, rng):
    n = len(perm); perm = list(perm); changed = False
    rk = 1.0 / max(n - 1, 1)
    for x in rng.permutation(n):
        x = int(x); cur = perm.index(x)
        base = perm[:cur] + perm[cur + 1:]; b = np.array(base); m = n - 1
        lo, hi = 0, m
        if pred[x] or succ[x]:
            posb = {y: i for i, y in enumerate(base)}
            for p_ in pred[x]: lo = max(lo, posb[p_] + 1)
            for s_ in succ[x]: hi = min(hi, posb[s_])
            if lo > hi: continue
        left = np.concatenate(([0.0], np.cumsum(S[b, x])))
        rcs = np.concatenate(([0.0], np.cumsum(S[x, b])))
        val = left + (rcs[-1] - rcs)
        d = np.zeros(m + 1)
        d[0] = A[x, b[0]]; d[m] = A[b[m - 1], x]
        if m > 1: d[1:m] = A[b[:-1], x] + A[x, b[1:]] - A[b[:-1], b[1:]]
        t = np.arange(m)
        cs0 = np.concatenate(([0.0], np.cumsum((t * rk - pp[b]) ** 2)))
        cs1 = np.concatenate(([0.0], np.cumsum(((t + 1) * rk - pp[b]) ** 2)))
        posval = cs0 + (cs1[-1] - cs1) + (np.arange(m + 1) * rk - pp[x]) ** 2
        first = np.full(m + 1, dep[b[0]]); first[0] = dep[x]
        val = val + lam * d - mu * posval - nu * first
        k = lo + int(np.argmax(val[lo:hi + 1]))
        if val[k] > val[cur] + 1e-9:
            perm = base[:k] + [x] + base[k:]; changed = True
    return perm, changed

def block_phase(perm, J, cons, max_len=4, max_rounds=3):
    perm = list(perm); n = len(perm); best = J(perm)
    for _ in range(max_rounds):
        improved = False
        for L in range(2, min(max_len, n - 1) + 1):
            for s in range(0, n - L + 1):
                block, rest = perm[s:s + L], perm[:s] + perm[s + L:]
                for k in range(len(rest) + 1):
                    if k == s: continue
                    cand = rest[:k] + block + rest[k:]
                    if violates(cand, cons): continue
                    v = J(cand)
                    if v > best + 1e-9: perm, best, improved = cand, v, True; break
                if improved: break
            if improved: break
        if not improved: break
    return perm

def order_sequence(S, A, pp, dep, cons, lam, mu, nu, restarts=6, seed=0, block=True, max_sweeps=30):
    n = S.shape[0]
    if n <= 2: return list(range(n)), 0.0
    rng = np.random.default_rng(seed)
    pred, succ = defaultdict(list), defaultdict(list)
    for a, b in (cons or []): succ[a].append(b); pred[b].append(a)
    J = lambda p: objective(p, S, A, pp, dep, lam, mu, nu)
    borda = S.sum(axis=1)
    r1 = np.argsort(np.argsort(-borda)); r2 = np.argsort(np.argsort(pp))
    inits = [list(np.argsort(-borda)), list(np.argsort(pp)), list(np.argsort(r1 + r2, kind="stable"))]
    while len(inits) < restarts: inits.append(list(rng.permutation(n)))
    best_perm, best_val = None, -np.inf
    for init in inits[:max(restarts, 3)]:
        perm = topo_repair([int(x) for x in init], cons, n)
        for _ in range(5):
            for _ in range(max_sweeps):
                perm, ch = insertion_pass(perm, S, A, pp, dep, lam, mu, nu, pred, succ, rng)
                if not ch: break
            if not block: break
            new = block_phase(perm, J, cons)
            if new == perm: break
            perm = new
        v = J(perm)
        if v > best_val: best_perm, best_val = perm, v
    return [int(x) for x in best_perm], float(best_val)

In [ ]:
# ===== CELL 6: sentence encoder, lexical features, ordered Hindi corpus (self-supervised training data) =====
from sentence_transformers import SentenceTransformer
LEX_DIM = 6
def lex_feats(mt):
    t = mt.split()
    if not t: return np.zeros(LEX_DIM, np.float32)
    return np.array([float(t[0] in ANAPHORIC), float(t[0] in CONNECTIVE), math.log1p(len(t)) / 4,
                     float(any(c.isdigit() for c in mt)), float(bool(re.search(r"[A-Za-z]", mt))),
                     float(t[-1] in FINITE_END)], np.float32)

enc_model = SentenceTransformer(CFG["enc_name"], device=DEVICE)
enc_model.max_seq_length = 64
if DEVICE == "cuda": enc_model.half()
EMB_DIM = enc_model.get_sentence_embedding_dimension()

def embed_texts(texts, batch=64):
    bs = batch
    while True:
        try:
            return enc_model.encode(texts, batch_size=bs, show_progress_bar=len(texts) > 2000,
                                    convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs > 8: bs //= 2; print("OOM under GPU cap -> batch", bs)
            else:
                print("GPU cap too small for encoder -> moving encoder to CPU"); enc_model.float().to("cpu"); bs = 32

def feats(model_texts): return np.hstack([embed_texts(model_texts), np.stack([lex_feats(t) for t in model_texts])])

X_SEG = {fn: feats([s["model_text"] for s in SEGS[fn]]) for fn in FILENAMES}
print("assignment segment features:", {fn: X_SEG[fn].shape for fn in FILENAMES})

# ---- ordered corpus: Hindi Wikipedia paragraphs (real, correctly ordered prose) ----
SENT_SPLIT = re.compile(r"(?<=[।?!])\s+|(?<=\.)\s+(?=[\u0900-\u097F])")
def deva_ratio(s):
    L = [c for c in s if c.isalpha()]
    return sum("\u0900" <= c <= "\u097f" for c in L) / max(len(L), 1)
def passage_from_paragraph(par, max_sents=12):
    out = []
    for s in SENT_SPLIT.split(par):
        s = " ".join(re.sub(r"\[[^\]]*\]", " ", re.sub(r"\([^)]*\)", " ", s)).split())
        mt = strip_for_model(s)
        if 4 <= len(mt.split()) <= 45 and deva_ratio(mt) >= 0.6: out.append(s)
    return out[:max_sents] if len(out) >= 5 else None

def build_corpus_hf(n_train, n_held, max_per_article=3, seed=0):
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", "20231101.hi", split="train", streaming=True)
    rng = random.Random(seed); train, held = [], []
    for ai, art in enumerate(ds):
        is_held = ai % 10 == 0                       # article-level split: no leakage between train and held-out
        if is_held and len(held) >= n_held:
            continue
        if not is_held and len(train) >= n_train:
            if len(held) >= n_held: break
            continue
        pars = [p for p in art["text"].split("\n") if len(p) > 150]; rng.shuffle(pars); got = 0
        for p in pars:
            ps = passage_from_paragraph(p)
            if ps: (held if is_held else train).append(ps); got += 1
            if got >= max_per_article: break
        if ai % 3000 == 0: print(f"  scanned {ai} articles | train {len(train)} | held-out {len(held)}")
    return train, held

def build_corpus_local(n_train, n_held):
    train, held = [], []
    for f in sorted(ORDERED_LOCAL_DIR.glob("*.txt")):
        for li, line in enumerate(f.read_text(encoding="utf-8-sig").splitlines()):
            ps = passage_from_paragraph(line)
            if ps: (held if (li % 10 == 0) else train).append(ps)
    return train[:n_train], held[:n_held]

def flatten(passages):
    sents, starts, lens = [], [], []
    for ps in passages: starts.append(len(sents)); lens.append(len(ps)); sents.extend(ps)
    return sents, np.array(starts, np.int64), np.array(lens, np.int64)

cache = CACHE_DIR / "ordered_feats.npz"
key = f"{CFG['n_train_passages']}-{CFG['n_heldout_passages']}-{CFG['enc_name']}-{EMB_DIM}"
loaded = False
if cache.exists():
    z = np.load(cache, allow_pickle=False)
    if str(z["key"]) == key:
        X_TR, TR_START, TR_LEN, X_HO, HO_START, HO_LEN = (z[k] for k in ("X_tr", "tr_start", "tr_len", "X_ho", "ho_start", "ho_len"))
        loaded = True; print("Loaded cached ordered-corpus features:", X_TR.shape, X_HO.shape)
if not loaded:
    try:
        train_p, held_p = build_corpus_hf(CFG["n_train_passages"], CFG["n_heldout_passages"])
    except Exception as ex:
        print("Hugging Face corpus unavailable (", str(ex)[:150], ")")
        if ORDERED_LOCAL_DIR.exists(): train_p, held_p = build_corpus_local(CFG["n_train_passages"], CFG["n_heldout_passages"])
        else: raise RuntimeError("No ordered Hindi corpus. Fix internet access or put *.txt (one paragraph/line) in ./ordered_hindi") from ex
    tr_s, TR_START, TR_LEN = flatten(train_p); ho_s, HO_START, HO_LEN = flatten(held_p)
    print(f"train passages {len(train_p)} ({len(tr_s)} sentences) | held-out passages {len(held_p)} ({len(ho_s)} sentences)")
    t0 = time.time()
    X_TR = feats([strip_for_model(s) for s in tr_s]); X_HO = feats([strip_for_model(s) for s in ho_s])
    print(f"embedded corpus in {time.time()-t0:.0f}s")
    np.savez(cache, key=key, X_tr=X_TR, tr_start=TR_START, tr_len=TR_LEN, X_ho=X_HO, ho_start=HO_START, ho_len=HO_LEN)
gpu_report("after encoding")
del enc_model; free_gpu()

In [ ]:
# ===== CELL 7: pairwise order / adjacency / position model on frozen sentence features =====
# 3-way pair head: A before B | A after B | unrelated (different passages); adjacency head: B immediately follows A;
# position head: relative position of a sentence inside its passage. Trained on correctly ordered Hindi Wikipedia passages.
class OrderNet(nn.Module):
    def __init__(self, d, hid=512):
        super().__init__()
        self.norm = nn.LayerNorm(4 * d)
        self.trunk = nn.Sequential(nn.Linear(4 * d, hid), nn.GELU(), nn.Dropout(0.2), nn.Linear(hid, 256), nn.GELU())
        self.order_head, self.adj_head = nn.Linear(256, 3), nn.Linear(256, 1)
        self.pos_net = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Linear(128, 1))
    def pair(self, xa, xb):
        h = self.trunk(self.norm(torch.cat([xa, xb, xa * xb, xa - xb], -1)))
        return self.order_head(h), self.adj_head(h).squeeze(-1)
    def pos(self, x): return torch.sigmoid(self.pos_net(x).squeeze(-1))

def sample_batch(rng, B, p_start, p_len, G=8, probs=(0.4, 0.4, 0.2)):
    P = len(p_len)
    kind = rng.choice(3, size=B, p=probs)                 # 0 = A before B, 1 = A after B, 2 = unrelated
    p = rng.integers(0, P, size=B); L = p_len[p]
    gap = np.where(rng.random(B) < 0.4, 1, rng.integers(2, G + 1, size=B)); gap = np.minimum(gap, L - 1)
    i = np.floor(rng.random(B) * (L - gap)).astype(np.int64); j = i + gap
    ia = np.where(kind == 0, i, j); ib = np.where(kind == 0, j, i)
    q = (p + rng.integers(1, P, size=B)) % P; Lq = p_len[q]
    unrel = kind == 2
    ia = np.where(unrel, np.floor(rng.random(B) * L).astype(np.int64), ia)
    ib = np.where(unrel, np.floor(rng.random(B) * Lq).astype(np.int64), ib)
    pb = np.where(unrel, q, p)
    adj = ((kind == 0) & (gap == 1)).astype(np.float32)
    ra = ia / np.maximum(L - 1, 1); rb = ib / np.maximum(p_len[pb] - 1, 1)
    return p_start[p] + ia, p_start[pb] + ib, kind, adj, ra.astype(np.float32), rb.astype(np.float32), gap

def auc_score(s, y):
    o = np.argsort(s); r = np.empty(len(s)); r[o] = np.arange(1, len(s) + 1)
    n1 = int((y == 1).sum()); n0 = len(y) - n1
    return float("nan") if n1 == 0 or n0 == 0 else float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

@torch.no_grad()
def eval_pairs(model, X, starts, lens, n=8000, seed=123):
    model.eval(); rng = np.random.default_rng(seed)
    ia, ib, kind, adj, ra, rb, gap = sample_batch(rng, n, starts, lens)
    Xt = torch.from_numpy(X)
    lo, la = model.pair(Xt[ia], Xt[ib]); pr = torch.softmax(lo, -1).numpy(); pa = torch.sigmoid(la).numpy()
    rel = kind < 2; pred_before = pr[:, 0] > pr[:, 1]
    res = dict(order_acc=float(np.mean(pred_before[rel] == (kind[rel] == 0))),
               related_vs_unrelated_acc=float(np.mean((pr.argmax(1) == 2) == (kind == 2))),
               adjacency_auc=auc_score(pa, adj))
    for name, m in (("gap1", gap == 1), ("gap2-3", (gap >= 2) & (gap <= 3)), ("gap4+", gap >= 4)):
        mm = rel & m; res["order_acc_" + name] = float(np.mean(pred_before[mm] == (kind[mm] == 0))) if mm.any() else float("nan")
    allx = np.arange(len(X)); pp = model.pos(Xt).numpy()
    rel_pos = np.concatenate([np.arange(l) / max(l - 1, 1) for l in lens]).astype(np.float32)
    res["position_corr"] = float(np.corrcoef(pp, rel_pos)[0, 1])
    return res

def train_order_model(steps, B, lr):
    model = OrderNet(X_TR.shape[1]).to(TRAIN_DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    Xt = torch.from_numpy(X_TR).to(TRAIN_DEVICE); rng = np.random.default_rng(CFG["seed"]); t0 = time.time()
    for step in range(1, steps + 1):
        model.train()
        ia, ib, kind, adj, ra, rb, _ = sample_batch(rng, B, TR_START, TR_LEN)
        xa, xb = Xt[torch.from_numpy(ia)], Xt[torch.from_numpy(ib)]
        lo, la = model.pair(xa, xb)
        loss = (F.cross_entropy(lo, torch.from_numpy(kind).long())
                + F.binary_cross_entropy_with_logits(la, torch.from_numpy(adj))
                + 0.5 * (F.mse_loss(model.pos(xa), torch.from_numpy(ra)) + F.mse_loss(model.pos(xb), torch.from_numpy(rb))))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if step % 500 == 0 or step == steps:
            ev = eval_pairs(model, X_HO, HO_START, HO_LEN)
            print(f"step {step:5d} loss {loss.item():.3f} | held-out order_acc {ev['order_acc']:.3f} "
                  f"adj_auc {ev['adjacency_auc']:.3f} rel_acc {ev['related_vs_unrelated_acc']:.3f} pos_corr {ev['position_corr']:.3f} ({time.time()-t0:.0f}s)")
    return model

MODEL_PATH = CACHE_DIR / "order_net.pt"
order_model = OrderNet(X_TR.shape[1])
if MODEL_PATH.exists() and CFG.get("reuse_trained", True):
    try: order_model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu")); print("Loaded cached order model")
    except Exception: order_model = train_order_model(CFG["train_steps"], CFG["batch"], CFG["lr"])
else: order_model = train_order_model(CFG["train_steps"], CFG["batch"], CFG["lr"])
torch.save(order_model.state_dict(), MODEL_PATH); order_model.eval()
HELDOUT_PAIR_METRICS = eval_pairs(order_model, X_HO, HO_START, HO_LEN)
print("Held-out Wikipedia pair metrics (chance: order_acc 0.5, adjacency_auc 0.5):")
for k, v in HELDOUT_PAIR_METRICS.items(): print(f"  {k:<28}{v:.3f}")

In [ ]:
# ===== CELL 8: score matrices, held-out ordering benchmark, weight selection =====
@torch.no_grad()
def prepare(X):
    n = X.shape[0]; Xt = torch.from_numpy(X)
    ii, jj = np.meshgrid(np.arange(n), np.arange(n), indexing="ij"); mask = ii != jj
    lo, la = order_model.pair(Xt[torch.from_numpy(ii[mask].astype(np.int64))], Xt[torch.from_numpy(jj[mask].astype(np.int64))])
    pr = torch.softmax(lo, -1).numpy(); pj = torch.sigmoid(la).numpy()
    Pb, Pa, Pu, Pj = (np.zeros((n, n)) for _ in range(4))
    Pb[mask], Pa[mask], Pu[mask], Pj[mask] = pr[:, 0], pr[:, 1], pr[:, 2], pj
    D = Pb - Pa; S = 0.5 * (D - D.T) * (2.0 / max(n - 1, 1))       # antisymmetric, scaled so terms stay comparable across n
    A = np.log(Pj + 1e-3); np.fill_diagonal(A, -10.0)
    pp = order_model.pos(Xt).numpy().astype(np.float64)
    dep = (X[:, EMB_DIM] + 0.5 * X[:, EMB_DIM + 1]).astype(np.float64)
    return dict(S=S, A=A, pp=pp, dep=dep, Pj=Pj, R=1 - 0.5 * (Pu + Pu.T))

def make_minifiles(n_files, k_pass=3, seed=7):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n_files):
        idx, pid, loc = [], [], []
        for q in rng.choice(len(HO_LEN), size=k_pass, replace=False):
            for t in range(int(HO_LEN[q])): idx.append(int(HO_START[q] + t)); pid.append(int(q)); loc.append(t)
        perm = rng.permutation(len(idx)); idx, pid, loc = np.array(idx)[perm], np.array(pid)[perm], np.array(loc)[perm]
        pre = prepare(X_HO[idx]); pre.update(pid=pid, loc=loc, k=k_pass); out.append(pre)
    return out

def metrics_for(perm, f):
    pid, loc = f["pid"][perm], f["loc"][perm]; taus = []
    for q in np.unique(pid):
        seq = loc[pid == q]
        if len(seq) > 1: taus.append(np.mean([np.sign(seq[j] - seq[i]) for i in range(len(seq)) for j in range(i + 1, len(seq))]))
    adj_ok = sum(1 for t in range(len(perm) - 1) if pid[t] == pid[t + 1] and loc[t + 1] == loc[t] + 1)
    return dict(tau=float(np.mean(taus)), adj=adj_ok / (len(perm) - f["k"]), first=float(loc[0] == 0),
                switches=int(np.sum(pid[1:] != pid[:-1])))

def eval_method(files, fn):
    ms = [metrics_for(fn(f), f) for f in files]
    return {k: float(np.mean([m[k] for m in ms])) for k in ms[0]}

HELDOUT_ORDER = {}
BEST = dict(lam=1.0, mu=10.0, nu=0.0)
if CFG["run_weight_search"]:
    files = make_minifiles(30); rng_b = np.random.default_rng(0)
    HELDOUT_ORDER["random shuffle"] = eval_method(files, lambda f: list(rng_b.permutation(len(f["pid"]))))
    HELDOUT_ORDER["position head only"] = eval_method(files, lambda f: list(np.argsort(f["pp"])))
    HELDOUT_ORDER["pairwise Borda only"] = eval_method(files, lambda f: list(np.argsort(-f["S"].sum(1))))
    grid = [(l, m, n_) for l in (0.0, 0.5, 2.0, 6.0) for m in (0.0, 10.0, 40.0) for n_ in (0.0, 3.0)]
    res = []; t0 = time.time()
    for l, m, n_ in grid:
        r = eval_method(files, lambda f: order_sequence(f["S"], f["A"], f["pp"], f["dep"], [], l, m, n_, restarts=2, seed=0, block=False)[0])
        res.append((r["tau"] + 0.5 * r["adj"], l, m, n_, r))
    res.sort(key=lambda x: -x[0])
    print(f"grid search {time.time()-t0:.0f}s (30 pseudo-files = 3 held-out Wikipedia passages shuffled together; ~24 sentences each)")
    print(f"{'lam':>5}{'mu':>6}{'nu':>5} | {'tau':>6}{'adjacency':>10}{'first':>7}{'switches':>9}")
    for sc, l, m, n_, r in res[:8]: print(f"{l:>5}{m:>6}{n_:>5} | {r['tau']:>6.3f}{r['adj']:>10.3f}{r['first']:>7.2f}{r['switches']:>9.1f}")
    BEST = dict(lam=res[0][1], mu=res[0][2], nu=res[0][3])
    HELDOUT_ORDER[f"full decoder {BEST} (insertion+block, 6 restarts)"] = eval_method(
        files, lambda f: order_sequence(f["S"], f["A"], f["pp"], f["dep"], [], **BEST, restarts=CFG["restarts"], seed=0, block=True)[0])
    print("\nHeld-out ORDERING benchmark on Wikipedia (ground truth known there; ideal switches = 2):")
    for k, v in HELDOUT_ORDER.items(): print(f"  {k:<58} tau {v['tau']:+.3f}  adjacency {v['adj']:.3f}  first-correct {v['first']:.2f}  switches {v['switches']:.1f}")
    print("These numbers describe Wikipedia, not the assignment files. The assignment has no ground truth.")
print("Chosen decoder weights:", BEST)

In [ ]:
# ===== CELL 9: order the seven assignment files =====
MANUAL_CONSTRAINTS = {}   # optional hard "A before B" constraints; snippets match the punctuation-stripped segment text, e.g.
# {"sample-4.txt": [("<Hindi snippet of sentence A>", "<Hindi snippet of sentence B>")]}
def resolve_constraints(fn, segs):
    cons = []
    for a, b in MANUAL_CONSTRAINTS.get(fn, []):
        ia = [s["id"] for s in segs if a in s["model_text"]]; ib = [s["id"] for s in segs if b in s["model_text"]]
        if len(ia) == 1 and len(ib) == 1 and ia[0] != ib[0]: cons.append((ia[0], ib[0]))
        else: print(f"WARNING {fn}: constraint snippet not unique/found -> skipped ({a[:20]!r}, {b[:20]!r})")
    return cons

ORDER = {}
for fn in FILENAMES:
    segs = SEGS[fn]; pre = prepare(X_SEG[fn]); cons = resolve_constraints(fn, segs)
    perm, J = order_sequence(pre["S"], pre["A"], pre["pp"], pre["dep"], cons, **BEST, restarts=CFG["restarts"], seed=CFG["seed"], block=True)
    padj = [float(pre["Pj"][perm[t], perm[t + 1]]) for t in range(len(perm) - 1)]
    pair_margin = [float(pre["S"][perm[t], perm[t + 1]] * (len(perm) - 1) / 2) for t in range(len(perm) - 1)]
    weak = sorted(range(len(padj)), key=lambda t: padj[t])[:max(3, len(padj) // 10)]
    low_conf = [dict(output_position=t + 2, segment_id=perm[t + 1], p_adjacent=round(padj[t], 4)) for t in sorted(weak)]
    ORDER[fn] = dict(perm=perm, objective=J, constraints=cons, p_adjacent=padj, pair_margin=pair_margin, low_confidence_transitions=low_conf,
                     opens_with_dependent=bool(pre["dep"][perm[0]] > 0), n_constraints=len(cons))
    print(f"{fn:<14} n={len(segs):>3}  J={J:8.2f}  mean P(adj)={np.mean(padj):.3f}  opens-with-dependent={ORDER[fn]['opens_with_dependent']}")

In [ ]:
# ===== CELL 10: write outputs =====
for sub in ("01_punctuated", "02_ordered", "03_order_maps", "04_audit"): (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
for fn in FILENAMES:
    segs, od = SEGS[fn], ORDER[fn]; stem = fn.replace(".txt", "")
    (OUT_DIR / "01_punctuated" / fn).write_text(" ".join(s["text"] for s in segs) + "\n", encoding="utf-8")
    (OUT_DIR / "02_ordered" / fn).write_text("\n".join(segs[i]["text"] for i in od["perm"]) + "\n", encoding="utf-8")
    mapping = dict(file=fn, n_segments=len(segs), segment_ids_are="0-based index in ORIGINAL (shuffled) order of the file",
                   objective=od["objective"], decoder_weights=BEST, hard_constraints=od["constraints"],
                   low_confidence_transitions=od["low_confidence_transitions"],
                   output_order=[dict(output_position=t + 1, original_segment_id=i, end_type=segs[i]["end_type"], flags=segs[i]["flags"],
                                      p_adjacent_to_previous=(round(od["p_adjacent"][t - 1], 4) if t > 0 else None), text=segs[i]["text"])
                                 for t, i in enumerate(od["perm"])])
    (OUT_DIR / "03_order_maps" / f"{stem}.order.json").write_text(json.dumps(mapping, ensure_ascii=False, indent=2), encoding="utf-8")
audit = {fn: dict(AUDIT[fn], segments=[{k: v for k, v in s.items() if k != "model_text"} for s in SEGS[fn]]) for fn in FILENAMES}
(OUT_DIR / "04_audit" / "segmentation_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2, default=float), encoding="utf-8")

tot = lambda k: sum(AUDIT[f][k] for f in FILENAMES)
lines = ["# Summary report", "",
 "## Pipeline", f"1. Normalisation: whitespace collapsed only; `NULL` preserved. 2. Punctuation: `{CFG['punct_model']}` (NULL removed from input, re-inserted). "
 f"3. Segmentation: model sentence-ends, then repair of over-long segments (>{CFG['soft_max_words']} words, forced above {CFG['hard_max_words']}) at the best-scoring boundary "
 "(model P(period) + finite-verb-final bonus + sentence-starter bonus + NULL bonus - function-word penalty). Repairs are heuristic and listed in the audit. "
 f"4. Ordering: frozen `{CFG['enc_name']}` sentence features + lexical features -> small pairwise/adjacency/position network trained on ordered Hindi Wikipedia paragraphs; "
 "global decoding by insertion + block-move local search.", "",
 "## Decoder objective", "J = sum over all pairs P(i before j) margin + lam*sum log P(adjacent) - mu*position-prior error - nu*dependent-opening penalty (+ optional hard constraints). "
 f"Weights {BEST} selected on held-out Wikipedia pseudo-files, not on the assignment data. J is a learned proxy for coherence; a high J does not prove a correct order.", "",
 "## Counts", f"- files: {len(FILENAMES)}; words: {tot('n_words')}; NULL tokens: {tot('n_null')}; segments: {tot('n_segments')}; "
 f"model boundaries: {tot('n_model_boundaries')}; repair splits: {tot('n_repair_splits')}", "",
 "| file | segments | repair splits | max words | opens with dependent-looking sentence |", "|---|---|---|---|---|"]
for fn in FILENAMES:
    a = AUDIT[fn]; lines.append(f"| {fn} | {a['n_segments']} | {a['n_repair_splits']} | {a['words_per_segment']['max']} | {ORDER[fn]['opens_with_dependent']} |")
lines += ["", "## Held-out validation (Wikipedia only; NOT assignment accuracy)", "```", json.dumps(HELDOUT_PAIR_METRICS, indent=1)]
for k, v in HELDOUT_ORDER.items(): lines.append(f"{k}: {json.dumps({a: round(b, 3) for a, b in v.items()})}")
lines += ["```", "", "## Limitations",
 "- No ground-truth order exists for the assignment files; no accuracy is claimed for them.",
 "- The order model is trained on encyclopedic prose; dialogue files (6-7) are out of domain and should be treated as low confidence.",
 "- Frozen sentence embeddings were trained for paraphrase, so they under-represent discourse cues (pronoun reference, connectives); lexical flags only partly compensate.",
 "- Segmentation repair is rule+model based; check `04_audit` for repaired boundaries, NULL contexts and flagged segments.",
 "- Files contain several topics/passages; the decoder is not told how many, and may interleave or mis-nest them."]
(OUT_DIR / "summary_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Wrote:", *sorted(str(p.relative_to(PROJECT_DIR)) for p in OUT_DIR.rglob("*") if p.is_file()), sep="\n  ")

In [ ]:
# ===== CELL 11: automated validation (re-reads the files written to disk) =====
checks = []
def chk(name, ok, detail=""): checks.append((name, bool(ok), detail)); print(("PASS " if ok else "FAIL ") + name + (f"  [{detail}]" if detail else ""))
strip_ins = lambda s: INSERTED_RE.sub(" ", s).split()
chk("all input files processed", len(FILENAMES) == 7 and all((OUT_DIR / "02_ordered" / f).exists() for f in FILENAMES), f"{len(FILENAMES)} files")
for fn in FILENAMES:
    n = len(SEGS[fn]); perm = ORDER[fn]["perm"]
    chk(f"{fn}: every segment used exactly once", sorted(perm) == list(range(n)), f"{n} segments")
    orig = INSERTED_RE.sub(" ", DOCS[fn]).split()
    punct_txt = (OUT_DIR / "01_punctuated" / fn).read_text(encoding="utf-8")
    ordered_lines = (OUT_DIR / "02_ordered" / fn).read_text(encoding="utf-8").splitlines()
    chk(f"{fn}: punctuated text == original words (only marks added)", strip_ins(punct_txt) == orig)
    ordered_tokens = [t for line in ordered_lines for t in strip_ins(line)]
    chk(f"{fn}: ordered file is a permutation of original words", Counter(ordered_tokens) == Counter(orig), f"{len(ordered_tokens)} vs {len(orig)}")
    chk(f"{fn}: NULL count preserved", ordered_tokens.count("NULL") == orig.count("NULL"), f"{orig.count('NULL')}")
    chk(f"{fn}: line count == segment count", len(ordered_lines) == n)
    js = json.loads((OUT_DIR / "03_order_maps" / f"{fn.replace('.txt', '')}.order.json").read_text(encoding="utf-8"))
    chk(f"{fn}: JSON order map matches file", [x["text"] for x in js["output_order"]] == ordered_lines)
    pos = {x: i for i, x in enumerate(perm)}
    chk(f"{fn}: hard constraints respected", all(pos[a] < pos[b] for a, b in ORDER[fn]["constraints"]), f"{ORDER[fn]['n_constraints']} constraints")
print("\nSEGMENT STATISTICS / FLAGS")
for fn in FILENAMES:
    segs = SEGS[fn]; a = AUDIT[fn]
    print(f"{fn}: words/segment {a['words_per_segment']} | flags {a['flag_counts']}")
    for s in sorted(segs, key=lambda s: -s["n_words"])[:2]: print(f"   longest ({s['n_words']}w, id {s['id']}): {s['text'][:110]}")
    for s in [s for s in segs if "short" in s["flags"]][:2]: print(f"   short ({s['n_words']}w, id {s['id']}): {s['text']}")
print("\nFIRST 3 SENTENCES OF EACH OUTPUT (eyeball; ⚠ = dependent-looking opening)")
for fn in FILENAMES:
    print(fn, "⚠" if ORDER[fn]["opens_with_dependent"] else "")
    for i in ORDER[fn]["perm"][:3]: print("   ", SEGS[fn][i]["text"][:120])
bad = [c for c in checks if not c[1]]
print("\n" + ("ALL AUTOMATED CHECKS PASSED (these verify integrity, NOT ordering correctness)." if not bad else f"{len(bad)} CHECK(S) FAILED: {[b[0] for b in bad]}"))
print("Held-out Wikipedia metrics are in outputs/summary_report.md; the assignment files have no ground truth.")

In [ ]:
# ===== T1: environment check (minimal changes to Kaggle's Python environment) =====
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
 
def _v(p):
    try: return version(p)
    except PackageNotFoundError: return None
def _t(s):
    out = []
    for part in (s or "0").split(".")[:3]:
        d = "".join(c for c in part if c.isdigit()); out.append(int(d) if d else 0)
    return tuple(out) + (0,) * (3 - len(out))
 
tf, hub, tk = _v("transformers"), _v("huggingface_hub"), _v("tokenizers")
print("installed -> transformers", tf, "| huggingface_hub", hub, "| tokenizers", tk)
env_ok = bool(tf and hub and (4, 40, 0) <= _t(tf) < (4, 58, 0) and (0, 30, 0) <= _t(hub) < (1, 0, 0))
if env_ok:
    print("Installed versions are inside the range known to work with IndicTrans2 remote code -> NOT reinstalling/downgrading.")
else:
    print("Versions outside the tested range -> pinning the combination that worked for you in Colab.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers==4.57.6", "tokenizers==0.22.2", "huggingface_hub==0.36.2"])
    raise RuntimeError("Pins installed. Restart the kernel (Run > Restart session), then run T1 again (it will pass) and continue. "
                       "Note: a restart clears variables; stage-2 outputs on disk are unaffected.")
 
def _try_import():
    try:
        try:
            from IndicTransToolkit.processor import IndicProcessor
        except ImportError:
            from IndicTransToolkit import IndicProcessor
        return IndicProcessor
    except Exception as e:
        return e
 
r = _try_import()
if isinstance(r, Exception):
    print("IndicTransToolkit not importable (", repr(r)[:150], ") -> installing it WITHOUT touching transformers/hub.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "IndicTransToolkit"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacremoses", "sentencepiece", "indic-nlp-library-itt", "regex", "nltk"])
    r = _try_import()
    if isinstance(r, Exception):
        raise RuntimeError(f"IndicTransToolkit still fails to import: {r!r}. Paste this error back to me.")
IndicProcessor = r
print("IndicProcessor OK | transformers", _v("transformers"), "| huggingface_hub", _v("huggingface_hub"), "| tokenizers", _v("tokenizers"))

In [ ]:
# ===== T2: configuration, paths, token, device =====
import os, re, json, time, math, gc, hashlib, zipfile, shutil, socket, warnings
from pathlib import Path
from collections import Counter
import numpy as np, torch
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
 
CFG = dict(
    mode="sentence_then_group",      # "sentence_then_group" (selected) | "paragraph_level" (strict: translate each 5-sentence group as one text)
    group_size=5,
    model_id="ai4bharat/indictrans2-indic-en-dist-200M",
    local_model_dir=None,            # e.g. "/kaggle/input/<your-model-dataset>" if you uploaded the model as a Kaggle dataset (no token/Internet needed)
    src_lang="hin_Deva", tgt_lang="eng_Latn",
    device_mode="cpu",               # "cpu" = fp32, matches the validated experiment. "gpu" = fp16 under the VRAM cap (faster; outputs may differ slightly)
    gpu_budget_gb=1.0,               # allocator cap; nvidia-smi additionally shows the CUDA context (~0.3-0.5 GB) -> total stays < 1.5 GB for this 211M model
    batch_size=4, num_beams=4, max_new_tokens=256,
    max_src_tokens=200,              # sources longer than this (after IndicProcessor tags) are split at clause boundaries, never truncated (model limit 256)
    null_mode="keep",                # "keep": send NULL tokens to the model as-is and check they survive | "drop": remove NULL from model input (source text keeps them)
    paragraph_sep="\n\n",
    stop_after_minutes=None,         # e.g. 40 -> stop cleanly after ~40 min; re-run T6 to resume
    reset=False,                     # True -> delete this mode's checkpoint and start over
)
G = CFG["group_size"]; SRC, TGT = CFG["src_lang"], CFG["tgt_lang"]
assert CFG["mode"] in ("sentence_then_group", "paragraph_level")
 
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
TR_ROOT = WORK / "translation"; TR_DIR = TR_ROOT / CFG["mode"]; TR_DIR.mkdir(parents=True, exist_ok=True)
CKPT = TR_DIR / "checkpoint.jsonl"
if CFG["reset"] and CKPT.exists(): CKPT.unlink(); print("checkpoint deleted")
D_TXT, D_JSON, D_VAL = TR_DIR / "translated_txt", TR_DIR / "bilingual_json", TR_DIR / "validation"
for d in (D_TXT, D_JSON, D_VAL): d.mkdir(exist_ok=True)
 
def get_hf_token():
    t = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if t: return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None
HF_TOKEN = get_hf_token()                       # never printed
print("HF token:", "found" if HF_TOKEN else "NOT found (needed only to download the gated model)")
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close(); print("Internet: reachable")
except Exception:
    print("Internet: NOT reachable (turn it on in Session options, or use CFG['local_model_dir'] with a Kaggle dataset)")
 
DEVICE = "cpu"
if CFG["device_mode"] == "gpu" and torch.cuda.is_available():
    try:
        _x = torch.ones(64, 64, device="cuda"); float((_x @ _x).sum()); del _x
        total = torch.cuda.get_device_properties(0).total_memory
        torch.cuda.set_per_process_memory_fraction(min(0.95, CFG["gpu_budget_gb"] * 1024**3 / total))
        DEVICE = "cuda"
    except Exception as ex:
        print("CUDA unusable (", str(ex)[:120], ") -> CPU")
torch.set_num_threads(max(1, os.cpu_count() or 4))
def free_gpu():
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
def gpu_report(tag=""):
    if DEVICE != "cuda": print(f"[{tag}] CPU mode"); return
    print(f"[{tag}] torch allocated {torch.cuda.memory_allocated()/2**30:.2f} GiB | peak {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
    try: print("   nvidia-smi:", subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"], text=True).strip())
    except Exception: pass
print("mode:", CFG["mode"], "| device:", DEVICE, "| output dir:", TR_DIR)

In [ ]:
# ===== T3: handoff - load the ordered Hindi sentences from stage 2 =====
def _numkey(p): return int(re.search(r"sample-(\d+)\.txt$", p.name).group(1))
def find_ordered_dir():
    cands = [Path.cwd() / "outputs" / "02_ordered", Path("/kaggle/working/outputs/02_ordered")]
    if Path("/kaggle/input").exists(): cands += sorted(Path("/kaggle/input").glob("**/02_ordered"))
    for c in cands:
        if c.is_dir() and list(c.glob("sample-*.txt")): return c
    return None
 
ORDERED_DIR = find_ordered_dir(); SRC_SENTS = {}
if ORDERED_DIR is not None:
    for p in sorted(ORDERED_DIR.glob("sample-*.txt"), key=_numkey):
        # only normalisation: strip each line and collapse inner whitespace; empty lines are dropped (stage 2 writes none)
        SRC_SENTS[p.name] = [" ".join(l.split()) for l in p.read_text(encoding="utf-8-sig").splitlines() if l.strip()]
    HANDOFF = f"files in {ORDERED_DIR}"
elif "ORDER" in globals() and "SEGS" in globals():
    for fn in FILENAMES: SRC_SENTS[fn] = [SEGS[fn][i]["text"] for i in ORDER[fn]["perm"]]
    HANDOFF = "in-memory ORDER/SEGS from stage 2"
else:
    raise FileNotFoundError("Stage-2 output not found. Expected /kaggle/working/outputs/02_ordered/sample-*.txt "
                            "(re-run stage 2 cell 10 or upload that folder as a dataset). If it lives elsewhere, send me that path.")
print("handoff:", HANDOFF)
if ORDERED_DIR is not None:                      # cross-check against the JSON order maps
    for fn, sents in SRC_SENTS.items():
        mp = ORDERED_DIR.parent / "03_order_maps" / (fn.replace(".txt", "") + ".order.json")
        if mp.exists():
            ref = [" ".join(x["text"].split()) for x in json.loads(mp.read_text(encoding="utf-8"))["output_order"]]
            if ref != sents: print(f"WARNING {fn}: 02_ordered text differs from 03_order_maps JSON")
print(f"{'file':<14}{'sentences':>10}{'groups(5)':>10}{'words':>8}{'NULL':>6}")
for fn, s in SRC_SENTS.items():
    print(f"{fn:<14}{len(s):>10}{math.ceil(len(s)/G):>10}{sum(len(x.split()) for x in s):>8}{sum(t == 'NULL' for x in s for t in x.split()):>6}")
print("TOTAL sentences", sum(len(v) for v in SRC_SENTS.values()), "| groups", sum(math.ceil(len(v)/G) for v in SRC_SENTS.values()),
      "(Marian run reported 320 / 67 on earlier provisional segmentation; a different number just means segmentation changed)")
if len(SRC_SENTS) != 7: print("WARNING: expected 7 files, found", len(SRC_SENTS))

In [ ]:
# ===== T3b: token + gated-access check (run after T2/T3, before T4) =====
import os
from huggingface_hub import HfApi

def _find_token():
    for name in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"):
        if os.environ.get(name): return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        for name in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "hf_token", "HF", "HUGGINGFACE"):
            try: return us.get_secret(name)
            except Exception: pass
        try: print("Secret labels visible to this notebook:", us.list_secrets())
        except Exception: pass
    except Exception as e:
        print("kaggle_secrets unavailable:", repr(e)[:100])
    return None

HF_TOKEN = _find_token()
if not HF_TOKEN:
    raise RuntimeError("Still no token. Add-ons -> Secrets: label must be exactly HF_TOKEN and toggled Attached, then re-run this cell.")
os.environ["HF_TOKEN"] = HF_TOKEN            # in-memory only; never printed

api = HfApi(token=HF_TOKEN)
try:
    print("Authenticated as:", api.whoami()["name"])
except Exception as e:
    raise RuntimeError("Token rejected (invalid/expired). Create a new READ token at huggingface.co/settings/tokens and update the secret.") from e
try:
    api.model_info("ai4bharat/indictrans2-indic-en-dist-200M")
    api.hf_hub_download("ai4bharat/indictrans2-indic-en-dist-200M", "config.json")
    print("Gated access to the model: OK")
except Exception as e:
    raise RuntimeError("Token works but this account has no access to the gated repo. Open "
                       "huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M while logged in as the SAME account and accept the terms.") from e

In [ ]:
# %%
# ===== T4: load IndicTrans2 (local snapshot, gated token from Kaggle Secrets) =====
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
 
if CFG["local_model_dir"]:
    MODEL_DIR = Path(CFG["local_model_dir"])
else:
    MODEL_DIR = Path("/tmp/hf_models") / CFG["model_id"].split("/")[-1]
    if not (MODEL_DIR / "config.json").exists():
        if not HF_TOKEN: print("No HF_TOKEN found: the download will fail if the model is gated for your account.")
        try:
            snapshot_download(repo_id=CFG["model_id"], local_dir=str(MODEL_DIR), token=HF_TOKEN,
                              ignore_patterns=["*.msgpack", "*.h5", "*.ot", "*.onnx"])   # plain download, no dry_run
        except Exception as ex:
            raise RuntimeError(f"Model download failed: {str(ex)[:300]}\nCheck: Internet On, secret HF_TOKEN attached, "
                               "gated-model terms accepted on the model page.") from ex
print("model dir:", MODEL_DIR)
 
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR), trust_remote_code=True,
                                              torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()
model.config.use_cache = False; model.generation_config.use_cache = False       # the working fix from Colab
ip = IndicProcessor(inference=True)
print("params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 1), "| device:", DEVICE); gpu_report("after load")
 
# ---- core translation primitives ----
@torch.no_grad()
def translate_batch(texts):
    pre = ip.preprocess_batch(texts, src_lang=SRC, tgt_lang=TGT)
    inputs = tokenizer(pre, truncation=True, max_length=256, padding="longest", return_tensors="pt", return_attention_mask=True).to(DEVICE)
    src_trunc = [int(n) >= 256 for n in inputs["attention_mask"].sum(1).tolist()]
    gen = model.generate(**inputs, num_beams=CFG["num_beams"], num_return_sequences=1,
                         max_new_tokens=CFG["max_new_tokens"], use_cache=False)
    eos = tokenizer.eos_token_id
    # decoder usually starts with the eos id, so look for a *later* eos; none means generation hit the length cap
    hit = [(not bool((row[1:] == eos).any())) if eos is not None else (gen.shape[1] > CFG["max_new_tokens"]) for row in gen]
    dec = tokenizer.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return ip.postprocess_batch(dec, lang=TGT), hit, src_trunc
 
def run_batch(texts):
    global DEVICE
    try:
        return translate_batch(texts)
    except torch.cuda.OutOfMemoryError:
        free_gpu()
        if len(texts) > 1:
            h = len(texts) // 2; a, b = run_batch(texts[:h]), run_batch(texts[h:])
            return a[0] + b[0], a[1] + b[1], a[2] + b[2]
        print("OOM at batch size 1 under the VRAM cap -> moving the model to CPU fp32")
        model.float().to("cpu"); DEVICE = "cpu"; free_gpu()
        return translate_batch(texts)
 
def src_token_len(text):
    pre = ip.preprocess_batch([text], src_lang=SRC, tgt_lang=TGT)
    return len(tokenizer([pre[0]], truncation=False, padding=False)["input_ids"][0])
 
# smoke test (checks the environment + use_cache fix; says nothing about quality)
_t0 = time.time(); _o = run_batch(["मथुरा जंक्शन एक प्रमुख रेलवे स्टेशन है।"])
print("smoke test ->", _o[0][0], f"({time.time()-_t0:.1f}s)")

In [ ]:
# %%
# ===== T5: units, NULL handling, safe splitting of over-long inputs, checkpoint helpers =====
DEVA = re.compile(r"[\u0900-\u097F]")
CONN = {"और", "तथा", "एवं", "लेकिन", "परंतु", "परन्तु", "जबकि", "क्योंकि", "जो", "कि", "जिसमें", "जहाँ", "जहां"}
 
def model_text(src):
    toks = src.split()
    if CFG["null_mode"] == "drop": toks = [t for t in toks if t != "NULL"]   # documented normalisation; the source text is stored untouched
    return " ".join(toks)
 
def split_to_fit(text):
    """Recursively split at the clause boundary nearest the middle until every piece fits max_src_tokens. Never truncates."""
    if src_token_len(text) <= CFG["max_src_tokens"]: return [text]
    w = text.split()
    if len(w) < 2: return [text]
    rng = range(3, len(w) - 2)
    strong = [i for i in rng if w[i - 1][-1:] in "।?!"]
    medium = [i for i in rng if w[i - 1][-1:] in ";:,-"]
    weak = [i for i in rng if w[i] in CONN]
    mid = len(w) / 2
    for c in (strong, medium, weak, list(rng)):
        if c: i = min(c, key=lambda k: abs(k - mid)); break
    else: i = len(w) // 2
    return split_to_fit(" ".join(w[:i])) + split_to_fit(" ".join(w[i:]))
 
def build_units():
    units = []
    for fn, sents in SRC_SENTS.items():
        if CFG["mode"] == "sentence_then_group":
            for i, s in enumerate(sents): units.append(dict(key=f"{fn}|{i}", file=fn, idx=i, sent_idx=[i], src=s))
        else:
            for p, st in enumerate(range(0, len(sents), G)):
                ix = list(range(st, min(st + G, len(sents))))
                units.append(dict(key=f"{fn}|p{p}", file=fn, idx=p, sent_idx=ix, src=" ".join(sents[i] for i in ix)))
    return units
 
sha1 = lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest()
FP = sha1(json.dumps([CFG["model_id"], CFG["num_beams"], CFG["max_new_tokens"], CFG["null_mode"], CFG["device_mode"], CFG["max_src_tokens"]]))[:12]
 
UNITS = build_units(); JOBS = []
for u in UNITS:
    mt = model_text(u["src"]); u["model_input"] = mt
    u["passthrough"] = not DEVA.search(mt)            # e.g. a sentence that is only "NULL" or a number: nothing to translate
    u["chunks"] = [] if u["passthrough"] else split_to_fit(mt)
    for ci, ch in enumerate(u["chunks"]):
        JOBS.append(dict(job=f"{u['key']}#{ci}", text=ch, sha=sha1(ch), ntok=src_token_len(ch)))
n_split = sum(len(u["chunks"]) > 1 for u in UNITS)
print(f"units {len(UNITS)} | model jobs {len(JOBS)} | passthrough units {sum(u['passthrough'] for u in UNITS)} | "
      f"units split for length {n_split} | max source tokens {max([j['ntok'] for j in JOBS] or [0])} (cap {CFG['max_src_tokens']}, model limit 256)")
 
def load_ckpt():
    done = {}
    if CKPT.exists():
        for line in CKPT.read_text(encoding="utf-8").splitlines():
            try: r = json.loads(line)
            except Exception: continue                 # a half-written last line after an interruption is ignored
            done[r["job"]] = r
    return done
sha_of = {j["job"]: j["sha"] for j in JOBS}
DONE = {k: r for k, r in load_ckpt().items() if sha_of.get(k) == r.get("sha") and r.get("cfg") == FP}
print(f"checkpoint: {len(DONE)}/{len(JOBS)} jobs already done (resume is automatic)")

In [ ]:
# %%
# ===== T6: translate (batched, checkpointed, resumable; progress bar) =====
pending = sorted([j for j in JOBS if j["job"] not in DONE], key=lambda j: -j["ntok"])   # longest first: memory problems show up early
t_start = time.time(); bs = CFG["batch_size"]; stopped = False
if not pending: print("Nothing pending.")
else:
    with open(CKPT, "a", encoding="utf-8") as fh:
        for b in tqdm(range(0, len(pending), bs), desc="batches"):
            if CFG["stop_after_minutes"] and (time.time() - t_start) / 60 > CFG["stop_after_minutes"]:
                stopped = True; print("Time budget reached: stopped cleanly. Re-run this cell to resume."); break
            batch = pending[b:b + bs]
            out, hit, trunc = run_batch([j["text"] for j in batch])
            for j, o, h, t in zip(batch, out, hit, trunc):
                rec = dict(job=j["job"], sha=j["sha"], cfg=FP, en=o, hit_cap=bool(h), src_trunc=bool(t))
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); DONE[j["job"]] = rec
            fh.flush(); os.fsync(fh.fileno())
    print(f"translated {len(DONE)}/{len(JOBS)} jobs in {(time.time()-t_start)/60:.1f} min"); gpu_report("after translation")
if len(DONE) < len(JOBS): print("NOT COMPLETE: run this cell again before T7.")

In [ ]:
# %%
# ===== T7: assemble units + per-unit quality flags =====
missing = [j["job"] for j in JOBS if j["job"] not in DONE]
if missing: raise RuntimeError(f"{len(missing)} jobs not translated yet; re-run T6.")
 
DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
def nums(s):
    s = re.sub(r"(?<=\d),(?=\d)", "", s.translate(DEV_DIGITS))
    return re.findall(r"\d+(?:\.\d+)?", s)
HARD = {"empty_translation", "untranslated_hindi", "generation_hit_cap", "source_truncated", "number_missing", "repetition"}
 
def unit_flags(u, en, gflags):
    f = list(gflags); src = u["src"]
    if not en.strip(): f.append("empty_translation")
    if DEVA.search(en): f.append(f"untranslated_hindi:{len(DEVA.findall(en))}_chars")
    miss = [n for n in dict.fromkeys(nums(src)) if n not in set(nums(en))]
    if miss: f.append("number_missing:" + ",".join(miss))
    n_null = sum(t == "NULL" for t in src.split())
    if CFG["null_mode"] == "keep" and en.split().count("NULL") != n_null: f.append(f"null_count_mismatch:src={n_null},en={en.split().count('NULL')}")
    sw = len([t for t in src.split() if t != "NULL"]); ew = len(en.split())
    if sw >= 4 and not (0.5 <= ew / max(sw, 1) <= 2.5): f.append(f"length_ratio:{ew/max(sw,1):.2f}")
    if en.strip() and en.rstrip()[-1:] not in ".?!\"')\u201d\u2019": f.append("no_terminal_punct")
    w = en.lower().split()
    if len(w) >= 9 and Counter(zip(w, w[1:], w[2:])).most_common(1)[0][1] >= 3: f.append("repetition")
    if CFG["mode"] == "paragraph_level":
        ne = len(re.findall(r"[.?!](?:\s|$)", en))
        if ne < len(u["sent_idx"]): f.append(f"fewer_sentences_than_source:en={ne},src={len(u['sent_idx'])}")
    return f
 
for u in UNITS:
    if u["passthrough"]:
        en, g = u["src"], ["passthrough_no_hindi"]
    else:
        recs = [DONE[f"{u['key']}#{ci}"] for ci in range(len(u["chunks"]))]
        en = " ".join(r["en"].strip() for r in recs); g = []
        if any(r["hit_cap"] for r in recs): g.append("generation_hit_cap")
        if any(r["src_trunc"] for r in recs): g.append("source_truncated")
        if len(recs) > 1: g.append(f"split_into_{len(recs)}_chunks")
    u["en"] = " ".join(en.split()); u["flags"] = unit_flags(u, u["en"], g)
is_hard = lambda fl: fl.split(":")[0] in HARD
n_hard = sum(any(is_hard(f) for f in u["flags"]) for u in UNITS); n_soft = sum(bool(u["flags"]) and not any(is_hard(f) for f in u["flags"]) for u in UNITS)
print(f"units: {len(UNITS)} | need-review (hard flags): {n_hard} | soft flags only: {n_soft} | clean: {len(UNITS)-n_hard-n_soft}")
for u in [u for u in UNITS if any(is_hard(f) for f in u["flags"])][:10]:
    print(f"  {u['file']} #{u['idx']}: {u['flags']} | {u['en'][:90]}")


In [ ]:
# %%
# ===== T8: write translated .txt files + bilingual JSON =====
NORMALISATION = ("Source = stage-2 ordered line, whitespace collapsed only. NULL tokens are "
                 + ("sent to the model unchanged and checked afterwards." if CFG["null_mode"] == "keep"
                    else "removed from the model input (source_hi keeps them; n_null_in_source records how many).")
                 + " Inputs above max_src_tokens are split at clause boundaries and re-joined (flag split_into_N_chunks). IndicProcessor applies its own standard pre/post-processing.")
SETTINGS = {k: CFG[k] for k in ("mode", "group_size", "model_id", "device_mode", "batch_size", "num_beams", "max_new_tokens", "max_src_tokens", "null_mode")}
SETTINGS["use_cache"] = False
 
for fn, sents in SRC_SENTS.items():
    us = [u for u in UNITS if u["file"] == fn]
    if CFG["mode"] == "sentence_then_group":
        sent_en = [u["en"] for u in us]
        paras = [dict(paragraph=p, sentence_indices=list(range(s, min(s + G, len(sent_en)))), english=" ".join(sent_en[s:s + G]))
                 for p, s in enumerate(range(0, len(sent_en), G))]            # last incomplete group is kept
        sent_rows = [dict(index=i, paragraph=i // G, source_hi=sents[i], model_input=u["model_input"], n_null_in_source=sum(t == "NULL" for t in sents[i].split()),
                          english=u["en"], flags=u["flags"]) for i, u in enumerate(us)]
    else:
        paras = [dict(paragraph=u["idx"], sentence_indices=u["sent_idx"], source_hi=u["src"], model_input=u["model_input"],
                      n_null_in_source=sum(t == "NULL" for t in u["src"].split()), english=u["en"], flags=u["flags"]) for u in us]
        sent_rows = [dict(index=i, paragraph=i // G, source_hi=sents[i], english=None) for i in range(len(sents))]
    (D_TXT / fn).write_text(CFG["paragraph_sep"].join(p["english"] for p in paras) + "\n", encoding="utf-8")
    (D_JSON / fn.replace(".txt", ".json")).write_text(json.dumps(
        dict(file=fn, mode=CFG["mode"], settings=SETTINGS, normalisation=NORMALISATION, handoff=HANDOFF,
             n_sentences=len(sents), n_paragraphs=len(paras), sentences=sent_rows, paragraphs=paras), ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote", len(list(D_TXT.glob("*.txt"))), "txt files and", len(list(D_JSON.glob("*.json"))), "JSON files in", TR_DIR)

In [ ]:
# %%
# ===== T9: validation (re-reads the files written to disk) + summary files =====
checks = []
def chk(name, ok, detail=""): checks.append(dict(name=name, ok=bool(ok), detail=detail)); print(("PASS " if ok else "FAIL ") + name + (f"  [{detail}]" if detail else ""))
 
per_file = {}; flagged = []
for fn, sents in SRC_SENTS.items():
    n = len(sents); exp_p = math.ceil(n / G)
    paras_txt = (D_TXT / fn).read_text(encoding="utf-8").rstrip("\n").split(CFG["paragraph_sep"])
    js = json.loads((D_JSON / fn.replace(".txt", ".json")).read_text(encoding="utf-8"))
    chk(f"{fn}: source sentence count == JSON n_sentences", js["n_sentences"] == n == len(js["sentences"]), f"{n}")
    chk(f"{fn}: source sentences unchanged and in stage-2 order", [s["source_hi"] for s in js["sentences"]] == sents)
    chk(f"{fn}: paragraph count == ceil(n/{G})", len(paras_txt) == exp_p == len(js["paragraphs"]), f"{exp_p}")
    chk(f"{fn}: final (possibly incomplete) group preserved", js["paragraphs"][-1]["sentence_indices"][-1] == n - 1, f"last group has {len(js['paragraphs'][-1]['sentence_indices'])} sentences")
    chk(f"{fn}: txt paragraphs == JSON paragraphs", paras_txt == [p["english"] for p in js["paragraphs"]])
    if CFG["mode"] == "sentence_then_group":
        chk(f"{fn}: every sentence has a translation", all(s["english"].strip() for s in js["sentences"]))
        chk(f"{fn}: paragraph == join of its sentence translations",
            all(p["english"] == " ".join(js["sentences"][i]["english"] for i in p["sentence_indices"]) for p in js["paragraphs"]))
        rows = js["sentences"]
    else:
        chk(f"{fn}: every group has a translation", all(p["english"].strip() for p in js["paragraphs"])); rows = js["paragraphs"]
    fl = Counter(f.split(":")[0] for r in rows for f in r["flags"])
    per_file[fn] = dict(sentences=n, paragraphs=exp_p, flag_counts=dict(fl),
                        hard_flagged_units=sum(any(is_hard(f) for f in r["flags"]) for r in rows))
    for r in rows:
        if r["flags"]:
            flagged.append(dict(file=fn, index=r.get("index", r.get("paragraph")), hard=any(is_hard(f) for f in r["flags"]), flags=r["flags"],
                                source=r["source_hi"][:160], english=(r["english"] or "")[:160]))
 
bad = [c for c in checks if not c["ok"]]; hard_list = [x for x in flagged if x["hard"]]
summary = dict(mode=CFG["mode"], settings=SETTINGS, handoff=HANDOFF, normalisation=NORMALISATION,
               totals=dict(files=len(SRC_SENTS), sentences=sum(len(v) for v in SRC_SENTS.values()),
                           paragraphs=sum(math.ceil(len(v) / G) for v in SRC_SENTS.values()), units_needing_review=len(hard_list), units_with_any_flag=len(flagged)),
               integrity_checks_failed=[c["name"] for c in bad], per_file=per_file, flagged_units=flagged,
               caveats=["Passing integrity checks means every sentence was translated, counted and written consistently. It does NOT measure translation quality.",
                        "Number check covers digits only (not number words). Untranslated-Hindi check looks for Devanagari characters in the output.",
                        "Length-ratio, terminal-punctuation, repetition and sentence-count flags are heuristics and can fire on correct output.",
                        "No reference translations exist, so no BLEU/chrF/COMET is reported. Read the flagged units and spot-check the rest.",
                        "Sentence-then-group is a methodological variant of 'translate groups of five'; its advantage was seen qualitatively on one railway passage only."])
(D_VAL / "validation_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
md = ["# Translation validation summary", "", f"Mode: `{CFG['mode']}` | model `{CFG['model_id']}` | device `{CFG['device_mode']}` | beams {CFG['num_beams']} | use_cache=False", "",
      "## Integrity", f"- checks run: {len(checks)}; failed: {len(bad)}", f"- sentences: {summary['totals']['sentences']}; paragraphs: {summary['totals']['paragraphs']}",
      "", "## Quality flags (heuristic)", f"- units needing review (hard flags): {len(hard_list)}; units with any flag: {len(flagged)}", "",
      "| file | sentences | paragraphs | hard-flagged | flag counts |", "|---|---|---|---|---|"]
md += [f"| {fn} | {d['sentences']} | {d['paragraphs']} | {d['hard_flagged_units']} | {d['flag_counts']} |" for fn, d in per_file.items()]
md += ["", "## Hard-flagged units", *[f"- {x['file']} #{x['index']}: {x['flags']} :: {x['english'][:100]}" for x in hard_list[:60]],
       "", "## Caveats", *[f"- {c}" for c in summary["caveats"]], "",
       "## Methodology", "Assignment wording: translate groups of five sentences. Default here: translate each sentence, then group five translations (final incomplete group kept). "
       "Strict alternative: set CFG['mode']='paragraph_level' and re-run T2-T10."]
(D_VAL / "validation_summary.md").write_text("\n".join(md) + "\n", encoding="utf-8")
print("\n" + ("ALL INTEGRITY CHECKS PASSED." if not bad else f"{len(bad)} INTEGRITY CHECK(S) FAILED: {[b['name'] for b in bad]}"),
      f"| units needing review: {len(hard_list)} | units with any flag: {len(flagged)}")
print("Integrity != quality: no reference translations exist; review the flagged units and spot-check the rest.")

In [ ]:
# %%
# ===== T10: final downloadable ZIP =====
ZIP_OUT = WORK / f"translation_outputs_{CFG['mode']}.zip"
with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(TR_DIR.rglob("*")):
        if p.is_file(): z.write(p, arcname=str(p.relative_to(TR_ROOT)))
with zipfile.ZipFile(ZIP_OUT) as z: names = z.namelist(); assert z.testzip() is None
print(f"{ZIP_OUT}  ({ZIP_OUT.stat().st_size/1024:.0f} KB, {len(names)} files)")
print(*names, sep="\n  ")
print("\nDownload: right sidebar -> Output -> /kaggle/working -> the .zip -> three dots -> Download.")
try:
    from IPython.display import FileLink, display; display(FileLink(str(ZIP_OUT.relative_to(Path("/kaggle/working"))) if str(ZIP_OUT).startswith("/kaggle/working") else str(ZIP_OUT)))
except Exception: pass
 